## Importing required libraries

In [0]:

from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, TimestampNTZType,DoubleType


In [0]:
%run "/Workspace/Users/abdulm63633@gmail.com/Ecommerce Lakehouse Project/01_setup_file/setup_utils"

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
dbutils.widgets.text("catalog", "ecommerce_lakehouse_project", "Catalog")
dbutils.widgets.text("data_source", "products", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://batch-ecommerce-lakehouse-project/{data_source}/landing/*.parquet'
print(base_path)

In [0]:
# Defining the schema to match your structure exactly
schema = StructType([
    StructField("product_id", LongType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("created_at", TimestampNTZType(), True)
])

df = (
    spark.read.format("parquet")
        .schema(schema)
        .load(base_path)
        .withColumn("read_timestamp", F.current_timestamp())
        .withColumn("batch_date", F.current_date())
        .select("*", "_metadata.file_name", "_metadata.file_size")

)

display(df.limit(10))

In [0]:
df.printSchema()
df.count()

In [0]:
df.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("append") \
    .saveAsTable(
        f"{catalog}.{bronze_schema}.{data_source}"
    )